# CUAD: LLM answers vs. gold answers, for one contract

One record, start to finish:

1. take one CUAD record and get its **context** (the contract text) and its **41 questions**,
2. let the LLM answer the questions from the context,
3. put each LLM answer next to the **gold answer** and judge it the way the official scorer does.

Needs `LLM_BASE_URL`, `LLM_API_KEY` and `TEXT_LLM_MODEL` (or `LLM_MODEL`) in `Project_1/.env`.

In [1]:
import html, importlib.util, json, os, time, urllib.error, urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/unified/dev.jsonl").exists())
DATA = ROOT / "data"
pd.set_option("display.max_colwidth", 90, "display.max_rows", 60)

for line in (ROOT / ".env").read_text(encoding="utf-8").splitlines():      # KEY=VALUE lines; the environment wins
    key, sep, value = line.strip().partition("=")
    if sep and key and not key.startswith("#"):
        os.environ.setdefault(key.strip(), value.strip().strip("'\""))

MODEL = os.environ.get("TEXT_LLM_MODEL") or os.environ.get("LLM_MODEL")
assert MODEL and os.environ.get("LLM_BASE_URL") and os.environ.get("LLM_API_KEY"), "fill in Project_1/.env first"
print("model:", MODEL)

model: claude-sonnet-5.5


## 1. One record: its context and its questions

The record in `dev.jsonl` holds the field names and the gold answers, but not the contract.
`pages` points to `raw/cuad/CUADv1.json`; the contract is the entry whose `title` is the `doc_id` without `cuad:`.
From that entry we take two things: `context` (the full contract text) and the 41 `question` texts.

Change `DOC_INDEX` (0 to 380) to try another contract.

In [2]:
DOC_INDEX = 0

cuad = [r for r in map(json.loads, (DATA / "unified/dev.jsonl").open(encoding="utf-8")) if r["source_dataset"] == "cuad"]
rec = cuad[DOC_INDEX]

raw = json.loads((DATA / rec["pages"][0]).read_text(encoding="utf-8"))["data"]
entry = next(d for d in raw if d["title"] == rec["doc_id"].split(":", 1)[1])["paragraphs"][0]

context = entry["context"]
questions = {qa["id"].rsplit("__", 1)[1]: qa["question"] for qa in entry["qas"]}      # clause name -> full question
questions = {name: questions[name] for name in rec["schema"]["properties"]}           # in the record's field order

print(rec["doc_id"])
print(f"context: {len(context):,} characters   questions: {len(questions)}\n")
print("CONTEXT (first 500 characters)\n" + context[:500] + "\n")
print("QUESTIONS (first 5)")
for name, q in list(questions.items())[:5]:
    print(f"- [{name}] {q}")

cuad:LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
context: 54,290 characters   questions: 41

CONTEXT (first 500 characters)
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an 

QUESTIONS (first 5)
- [Document Name] Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
- [Parties] Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
- [Agreemen

## 2. The LLM answers the questions from the context

One call. The model gets every question and the whole context, and returns for each clause the passages it would highlight,
copied word for word, plus how confident it is. An empty list means "the contract has nothing on this".

In [3]:
PROMPT = """Below are {n} questions about one contract, followed by the contract.
Answer each question by quoting the relevant parts of the contract.

Reply with ONE JSON object and nothing else, with one entry per question name:
{{"<name>": {{"spans": ["<exact quote>", ...], "confidence": <0..1>}}, ...}}

Rules:
- Copy every span character for character from the contract. Never paraphrase or summarise.
- If several separate passages answer a question, list each one as its own span.
- For Document Name, Parties and dates, quote only the name or the date. For other clauses, quote the whole sentence or clause.
- If the contract has nothing for a question, return {{"spans": [], "confidence": 0}}. That is the usual case.
- Confidence is the probability that your spans are exactly what a careful lawyer would highlight.

QUESTIONS
{questions}

CONTRACT
{context}
"""

def call_llm(prompt: str, tries: int = 6) -> str:
    body = json.dumps({"model": MODEL, "max_tokens": 16000, "messages": [{"role": "user", "content": prompt}]}).encode()
    req = urllib.request.Request(os.environ["LLM_BASE_URL"].rstrip("/") + "/chat/completions", body,
                                 {"Content-Type": "application/json", "Authorization": f"Bearer {os.environ['LLM_API_KEY']}"})
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(req, timeout=600) as r:
                return json.load(r)["choices"][0]["message"]["content"]
        except urllib.error.HTTPError as e:
            detail = e.read().decode(errors="replace")[:300]
            if e.code == 429 and attempt < tries - 1:          # the gateway's limit resets every minute
                print(f"rate limited, waiting 30 s (attempt {attempt + 1} of {tries})")
                time.sleep(30)
                continue
            raise RuntimeError(f"HTTP {e.code}: {detail}") from None

prompt = PROMPT.format(n=len(questions), context=context,
                       questions="\n".join(f"- {name}: {q}" for name, q in questions.items()))
reply = call_llm(prompt)
llm = json.loads(reply[reply.index("{"): reply.rindex("}") + 1])

# One answer per clause, in the scorer's format: spans joined with " | ", or None when the model found nothing.
answers, confidence = {}, {}
for name in questions:
    a = llm.get(name) if isinstance(llm.get(name), dict) else {}
    spans = [s for s in (a.get("spans") or []) if isinstance(s, str) and s.strip()]
    answers[name] = " | ".join(spans) or None
    confidence[name] = float(a.get("confidence") or 0) if spans else None

print(f"the model answered {sum(v is not None for v in answers.values())} of {len(answers)} questions "
      f"and left {sum(v is None for v in answers.values())} empty")

rate limited, waiting 30 s (attempt 1 of 6)
the model answered 24 of 41 questions and left 17 empty


## 3. Compare with the gold data

`rec["gold"]` has the lawyers' answer for each clause: the highlighted passages joined with ` | `, or `null` when the clause is absent.

The verdict uses the scorer's own function, `span_f1` from `5_score.py`: both texts are split into words, and

- **precision** = share of the LLM's words that are also in gold,
- **recall** = share of gold's words that the LLM also has,
- **token F1** combines the two; the answer counts as **correct at 0.8 or more**.

| Outcome | LLM answer | Gold |
| --- | --- | --- |
| correct | given, F1 ≥ 0.8 | present |
| wrong | given, F1 < 0.8 | present |
| hallucinated | given | null |
| missed | empty | present |
| true null | empty | null |

In [4]:
spec = importlib.util.spec_from_file_location("score", ROOT / "5_score.py")
score = importlib.util.module_from_spec(spec); spec.loader.exec_module(score)

def judge(pred, gold):
    if pred is None:
        return ("missed" if gold is not None else "true null"), None
    if gold is None:
        return "hallucinated", None
    f1 = score.span_f1(pred, gold)
    return ("correct" if f1 >= score.SPAN_F1_THRESHOLD else "wrong"), round(f1, 2)

tidy = lambda v: " ".join(v.split()) if isinstance(v, str) else None
gold = rec["gold"]
table = pd.DataFrame([
    {"clause": name, "LLM answer": tidy(answers[name]), "gold answer": tidy(gold[name]),
     "outcome": judge(answers[name], gold[name])[0], "token F1": judge(answers[name], gold[name])[1],
     "LLM confidence": confidence[name]}
    for name in questions
]).set_index("clause")

summary = table["outcome"].value_counts().reindex(["correct", "wrong", "hallucinated", "missed", "true null"], fill_value=0)
answered = summary["correct"] + summary["wrong"] + summary["hallucinated"]
print(summary.to_string())
print(f"\nprecision (correct / answered): {summary['correct']}/{answered} = {summary['correct'] / max(answered, 1):.0%}")
print(f"gold answers found correctly:   {summary['correct']}/{table['gold answer'].notna().sum()}")

outcome
correct          9
wrong           10
hallucinated     5
missed           0
true null       17

precision (correct / answered): 9/24 = 38%
gold answers found correctly:   9/19


### Every clause side by side

Clauses where both the LLM and gold are empty (true nulls) are left out to keep the table short.

In [5]:
COLORS = {"correct": "#d9f2e3", "wrong": "#fde2d6", "hallucinated": "#f9d0d0", "missed": "#fff3c9"}
shown = table[table["outcome"] != "true null"].fillna({"LLM answer": "(empty)", "gold answer": "(null)"})
(shown.style
      .apply(lambda row: [f"background-color: {COLORS[row['outcome']]}; color: #0b0b0b"] * len(row), axis=1)
      .format({"token F1": "{:.2f}", "LLM confidence": "{:.2f}"}, na_rep="")
      .set_properties(subset=["LLM answer", "gold answer"], **{"max-width": "360px", "white-space": "normal", "text-align": "left"}))

,LLM answer,gold answer,outcome,token F1,LLM confidence
clause,,,,,
Document Name,DISTRIBUTOR AGREEMENT,DISTRIBUTOR AGREEMENT,correct,1.00,0.90
Parties,Electric City Corp. | Electric City of Illinois LLC,Distributor | Electric City Corp. | Electric City of Illinois L.L.C. | Company | Electric City of Illinois LLC,wrong,0.64,0.80
Agreement Date,"7th day of September, 1999","7th day of September, 1999.",correct,0.80,0.90
Effective Date,"subject to the commencement date established in Section 1.3, shall be effective immediately.","The term of this Agreement shall be ten (10) years (the ""Term"") which shall commence on the date upon which the Company delivers to Distributor the last Sample, as defined hereinafter. | Unless earlier terminated otherwise provided therein, this Agreement, subject to the commencement date established in Section 1.3, shall be effective immediately.",wrong,0.39,0.40
Expiration Date,"The term of this Agreement shall be ten (10) years (the ""Term"") which shall commence on the date upon which the Company delivers to Distributor the last Sample, as defined hereinafter.","The term of this Agreement shall be ten (10) years (the ""Term"") which shall commence on the date upon which the Company delivers to Distributor the last Sample, as defined hereinafter.",correct,1.00,0.60
Renewal Term,"If Distributor complies with all of the terms of this Agreement, the Agreement shall be renewable on an annual basis for one (1) year terms for up to another ten (10) years on the same terms and conditions as set forth herein.","If Distributor complies with all of the terms of this Agreement, the Agreement shall be renewable on an annual basis for one (1) year terms for up to another ten (10) years on the same terms and conditions as set forth herein.",correct,1.00,0.80
Governing Law,This Agreement is to be construed according to the laws of the State of Illinois.,This Agreement is to be construed according to the laws of the State of Illinois.,correct,1.00,0.90
Non-Compete,nor will Distributor solicit any customer or potential customer of Company to purchase a competitive product during that period.,(null),hallucinated,,0.30
Exclusivity,Company hereby appoints Distributor as Company's exclusive distributor within the Market and grants to Distributor the exclusive right to sell and distribute Products within the Market | The Distributor shall not order or purchase Products from any source other than the Company.,"The Distributor shall not order or purchase Products from any source other than the Company. | Company hereby appoints Distributor as Company's exclusive distributor within the Market and grants to Distributor the exclusive right to sell and distribute Products within the Market, and Distributor hereby accepts such appointment and such grant, in accordance with the terms and conditions of this Agreement. | The Company appoints the Distributor as an exclusive distributor of Products in the Market, subject to the terms and conditions of this Agreement.",wrong,0.65,0.60


### Why the wrong ones are wrong

For each answer that did not match, the full texts with the words they share in **bold**.
Text that is bold on both sides is where they agree; plain text on the gold side is what the LLM left out, and plain text
on the LLM side is what it quoted that the lawyers did not.

In [ ]:
from collections import Counter

def mark(text: str, other: str) -> str:
    """HTML of `text` with the words that also occur in `other` in bold (same word matching as the scorer)."""
    budget = Counter(score.norm_string(other).split())
    out = []
    for word in text.split():
        key = score.norm_string(word)
        if budget[key] > 0:
            budget[key] -= 1
            out.append(f"<b>{html.escape(word)}</b>")
        else:
            out.append(f"<span style='color:#898781'>{html.escape(word)}</span>")
    return " ".join(out)

blocks = []
for name, row in table[table["outcome"].isin(["wrong", "hallucinated", "missed"])].iterrows():
    pred, g = answers[name], gold[name]
    f1 = "" if pd.isna(row["token F1"]) else f", token F1 {row['token F1']:.2f}"
    blocks.append(
        f"<h4 style='margin:18px 0 4px'>{html.escape(name)} <span style='font-weight:normal'>({row['outcome']}{f1})</span></h4>"
        f"<p style='margin:2px 0'><i>Question:</i> {html.escape(questions[name].split('Details:', 1)[-1].strip())}</p>"
        f"<p style='margin:6px 0'><i>LLM:</i> {mark(pred, g or '') if pred else '(empty)'}</p>"
        f"<p style='margin:6px 0'><i>Gold:</i> {mark(g, pred or '') if g else '(null: the clause is not in the contract)'}</p>")
display(HTML("<div style='max-width:900px;line-height:1.45'>" + "".join(blocks) + "</div>") if blocks else HTML("<p>Nothing to show: every answer matched.</p>"))